# Carbon Pricing and Human Capital Project: Dataset (Cleaning Pipeline)
### Task: Reshaping and Aggregating World Bank Carbon Pricing Data (2015-2025)

**Research Scope:** Germany, Austria, Switzerland, Norway, Finland, and Sweden.

This Notebook automates the extraction of nominal carbon values from the World Bank's wide-format layout. 
This Notebook:
1. Filters for the  target countries
2. Shapes the time-series variables from horizontal years into a vertical panel format (long format)
3. Combines multiple concurrent policies (such as local carbon taxes operating alongside the EU ETS framework) into a singular, unified metric: `Total_Carbon_Price_EUR`.


In [3]:
import pandas as pd
import numpy as np

# Defining relative file paths
raw_file_path = "../data/raw/worldbank_carbon_pricing_raw.xlsx"
output_file_path = "../data/processed/carbon_prices_clean.csv"

print("Testing raw spreadsheet read with proper row positioning...")
# header=5 tells pandas that the true column headers are on row 5 (0-indexed as line 5)
df_preview = pd.read_excel(raw_file_path, sheet_name="Compliance_Price", header=5)
df_preview.head(3)

Testing raw spreadsheet read with proper row positioning...


,Tax_AN,Andorra carbon tax,Carbon tax,Europe & Central Asia,High income,US$/tCO2e,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,0.18,0.19,0.20,0.21,0.22,33.156,32.625,32.247,32.364,34.815
0,Tax_AR,Argentina carbon tax,Carbon tax,Latin America & Caribbean,Upper middle income,US$/tCO2e,NaN,NaN,NaN,NaN,...,NaN,8.914348,6.187569,6.342015,5.321599,4.995626,3.218257,0.811554,5.327915,4.027281
1,ETS_AU1,Australia CPM,ETS,East Asia & Pacific,High income,US$/tCO2e,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,ETS_AU2,Australia Safeguard Mechanism,ETS,East Asia & Pacific,High income,US$/tCO2e,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,21.903750,21.816120,24.676225


In [13]:
target_countries_list = ["Germany", "Austria", "Switzerland", "Norway", "Finland", "Sweden"]
target_years = [str(year) for year in range(2015, 2026)]

# 1. Read sheet from row index level 1 (bypassing title row)
df_raw = pd.read_excel(raw_file_path, sheet_name="Compliance_Price", header=1)

# 2. DYNAMICALLY search for the row index that contains the actual header text
true_header_row_index = None
for idx, row in df_raw.iterrows():
    row_str = row.astype(str).str.lower().values
    if any("unique id" in s or "instrument name" in s or "jurisdiction" in s for s in row_str):
        true_header_row_index = idx
        break

if true_header_row_index is not None:
    # Promote that matching row to be the column headers
    df_raw.columns = df_raw.iloc[true_header_row_index].astype(str).str.strip()
    df_raw = df_raw.iloc[true_header_row_index + 1:].reset_index(drop=True)
else:
    # Structural fallback if row promote fails
    df_raw.columns = [str(col).strip() for col in df_raw.columns]

# 3. Locate the country data column dynamically by matching your exact target country arrays
juris_col = None
for col in df_raw.columns:
    match_count = df_raw[col].astype(str).str.contains('|'.join(target_countries_list), case=False, na=False).sum()
    if match_count > 0:
        juris_col = col
        break

print(f"🎯 Verified structural country tracker column: '{juris_col}'")

# 4. Filter rows for target countries
df_filtered = df_raw[df_raw[juris_col].astype(str).str.contains('|'.join(target_countries_list), case=False, na=False)].copy()

# 5. Melt table from Wide to Long panel format
id_variables = [col for col in df_filtered.columns if col not in target_years]
available_years = [col for col in df_filtered.columns if col in target_years]

df_long = pd.melt(df_filtered, id_vars=id_variables, value_vars=available_years, var_name="Year", value_name="Nominal_Price")

# 6. Standardise country names
def map_to_country(text):
    for c in target_countries_list:
        if c.lower() in str(text).lower():
            return c
    return np.nan

df_long["Country"] = df_long[juris_col].apply(map_to_country)
df_long = df_long.dropna(subset=["Country"])

# 7. Convert nominal pricing parameters and sum concurrent taxes/ETS lines
df_long["Nominal_Price"] = pd.to_numeric(df_long["Nominal_Price"], errors="coerce").fillna(0.0)
df_long["Year"] = df_long["Year"].astype(int)

df_panel = df_long.groupby(["Country", "Year"], as_index=False)["Nominal_Price"].sum()
df_panel = df_panel[(df_panel["Year"] >= 2015) & (df_panel["Year"] <= 2025)]
df_panel = df_panel.sort_values(by=["Country", "Year"]).reset_index(drop=True)
df_panel.columns = ["Country", "Year", "Total_Carbon_Price_EUR"]

# Display the panel result directly inside the notebook
df_panel


🎯 Verified structural country tracker column: 'Name of the initiative'


,Country,Year,Total_Carbon_Price_EUR
0,Austria,2015,0.000000
1,Austria,2016,0.000000
2,Austria,2017,0.000000
3,Austria,2018,0.000000
4,Austria,2019,0.000000
...,...,...,...
61,Switzerland,2021,147.577602
62,Switzerland,2022,195.449696
63,Switzerland,2023,224.619327
64,Switzerland,2024,191.295458


In [14]:
df_panel.to_csv(output_file_path, index=False)
print(f"Success! Dataset A is officially processed and saved to: {output_file_path}")

Success! Dataset A is officially processed and saved to: ../data/processed/carbon_prices_clean.csv
